> **Experiment E00: Data cache**  
> Spec, decision rule and results: [docs/experiments/E00_data_cache.md](../docs/experiments/E00_data_cache.md)  
> Depends on: none · Status: done  
> Registered in `experiments/registry.yaml`; see the [experiment graph](../docs/experiments/EXPERIMENT_GRAPH.md).

# CARE to Compare — Wind Farm C — **Cache builder**

Run this **once, on a CPU-only session** (Settings → Accelerator → None). CPU sessions do not draw against your 30 GPU-hours.

It parses all 58 event CSVs — reading only the ~226 columns the models actually use instead of all ~957 — and writes them as parquet.

**Then:** Save Version → the output appears under *Output*. Publish it as a Dataset, attach that dataset to both GPU notebooks, and they will detect it automatically via `care_cache_manifest.json` and skip parsing entirely.

Re-run this only if you change `USE_AVG_ONLY` or `DROP_REACTIVE` — the notebooks refuse a cache whose column set doesn't match.

## Shared setup
Identical to the two training notebooks.

In [ ]:
# =============================================================================
# CONFIG  --  identical in both notebooks. Do not change one without the other.
# =============================================================================
import os, gc, re, json, math, time, hashlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- scope -----------------------------------------------------------------
FARM               = "Wind Farm C"     # FAQ: status labels are only evaluation-
                                       # usable for Farms B and C. Farm A differs.

# ---- features (FAQ: "Avg features are often the safest starting point";
#      Min/Max/Std contain implausible values) ---------------------------------
USE_AVG_ONLY       = True
DROP_REACTIVE      = False             # set True to drop reactive_power_* sensors
ANGLE_TRANSFORM    = True              # FAQ lists angle transformation as a
                                       # typical preprocessing step
MAX_NAN_FRACTION   = 0.30              # drop a feature if >30% NaN in training

# ---- normal-behaviour filter (FAQ: filtering by status_type_id is intended
#      use; power-curve filtering is "also reasonable") -----------------------
NORMAL_STATUS_IDS      = {0, 2}
WIND_SPEED_CUTIN       = 3.0
WIND_SPEED_RATED       = 12.0
POWER_NEAR_ZERO_THRESH = 0.02

# ---- model (paper Table 3, Wind Farm C) ------------------------------------
HIDDEN_LAYERS      = [133, 83, 20, 83, 133]
LEARNING_RATE      = 0.0056            # paper Table 3
USE_LR_PLATEAU     = True
# At the paper's LR the TRAINING loss increases across epochs while validation
# plateaus -- the optimiser is overshooting, and early stopping then restores an
# under-trained model. ReduceLROnPlateau keeps the paper's starting LR but lets
# it decay when validation stalls. Set False to reproduce the paper exactly.
LR_PLATEAU_FACTOR  = 0.5
LR_PLATEAU_PATIENCE = 4
NOISE_STD          = 0.0               # no denoising noise for Farm C
BATCH_SIZE         = 64
MAX_EPOCHS         = 200
EARLY_STOP_PATIENCE = 10               # deviation from the 3 used earlier:
                                       # 3 stops on noise before convergence
VAL_FRACTION       = 0.15              # held out for early stopping AND for
                                       # threshold calibration (never trained on)
VAL_SPLIT_MODE     = "chronological"   # "chronological" | "random"
# A RANDOM split interleaves validation rows between training rows minutes
# apart, so validation error measures within-period noise only. The threshold it
# yields is then far too low for a prediction window that sits months later, and
# the model over-flags. A CHRONOLOGICAL split (last VAL_FRACTION of the training
# period) makes the held-out rows temporally downstream, so the threshold
# absorbs some seasonal drift. Switch to "random" to see the difference.
SEED               = 42

# ---- thresholding ----------------------------------------------------------
# FAQ: "for early fault detection, you want to predict the event_label" -- NOT
# status_type_id. A normal-behaviour model has no anomaly labels at fit time, so
# the threshold is a high quantile of reconstruction error on held-out NORMAL
# data. Sweep SCORE_QUANTILE to trade Coverage against Accuracy.
SCORE_QUANTILE        = 0.99
CRITICALITY_THRESHOLD = 72             # FAQ: the documented default
BETA                  = 0.5
CARE_WEIGHTS          = (1, 1, 1, 2)   # (Coverage, Earliness, Reliability, Accuracy)

DTYPE = np.float32   # sensor dtype, used by every reader

np.random.seed(SEED)
print("Config loaded. Farm:", FARM)

In [ ]:
# =============================================================================
# DATA DISCOVERY
# =============================================================================
DATASET_ROOT = "/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection"

def find_farm_dir(root, farm_name):
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == farm_name and "datasets" in dirnames:
            return dirpath
    raise FileNotFoundError(f"Could not locate '{farm_name}' under {root}")

FARM_DIR     = find_farm_dir(DATASET_ROOT, FARM)
DATASETS_DIR = os.path.join(FARM_DIR, "datasets")

def first_existing(*cands):
    for c in cands:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(cands)

event_info = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_event_info.csv"),
    os.path.join(FARM_DIR, "event_info.csv")))
feature_description = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_feature_description.csv"),
    os.path.join(FARM_DIR, "feature_description.csv")))

event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],   errors="coerce")

# The dataset ships each file twice (comma- and semicolon-delimited). Pick one
# variant so no event is loaded twice.
all_csvs = [f for f in os.listdir(DATASETS_DIR) if f.endswith(".csv")]
has_comma = any(f.startswith("comma_") for f in all_csvs)
has_plain = any(not f.startswith("comma_") for f in all_csvs)
if has_comma and has_plain:
    csv_files, DATA_SEP = [f for f in all_csvs if f.startswith("comma_")], ","
else:
    csv_files, DATA_SEP = all_csvs, ("," if has_comma else ";")

def extract_dataset_id(fname):
    m = re.search(r"\d+", fname)
    if not m:
        raise ValueError(f"No numeric id in {fname}")
    return int(m.group())

csv_files = sorted(csv_files, key=extract_dataset_id)
ids = [extract_dataset_id(f) for f in csv_files]
assert len(ids) == len(set(ids)), "Duplicate dataset ids after variant filtering"

n_anom   = (event_info["event_label"] == "anomaly").sum()
n_normal = (event_info["event_label"] == "normal").sum()
print(f"{len(csv_files)} event datasets | event_info: {n_anom} anomaly / {n_normal} normal")

# asset_id -> which physical turbine each event belongs to. Needed for the
# leakage diagnostics and for the pooled notebook's asset-disjoint split.
# Farm C's event_info has no asset_id column, but every DATA file carries one.
# Read a single cell from each file -- cheap, and without it we cannot tell
# which events share a turbine.
def build_event_to_asset():
    if "asset_id" in event_info.columns:
        return dict(zip(event_info["event_id"].astype(int),
                        event_info["asset_id"].astype(int)))
    m = {}
    for f in csv_files:
        try:
            v = pd.read_csv(os.path.join(DATASETS_DIR, f), sep=DATA_SEP,
                            usecols=["asset_id"], nrows=1)["asset_id"].iloc[0]
            m[extract_dataset_id(f)] = int(v)
        except Exception as e:
            print(f"  could not read asset_id from {f}: {e}")
    return m

EVENT_TO_ASSET = build_event_to_asset()
if EVENT_TO_ASSET:
    n_assets = len(set(EVENT_TO_ASSET.values()))
    print(f"{n_assets} distinct turbines across {len(csv_files)} events "
          f"({len(csv_files) - n_assets} events share a turbine with another event)")
else:
    print("WARNING: no asset_id anywhere -- effective-N and the pooled split "
          "will fall back to event-level handling")

In [ ]:
# =============================================================================
# FEATURE SELECTION  --  Avg-only, optional angle transform
# =============================================================================
DESCRIPTIVE_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]

_sample = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP, nrows=200)
ALL_SENSOR_COLS = [c for c in _sample.columns if c not in DESCRIPTIVE_COLS]

if USE_AVG_ONLY:
    SENSOR_COLS = [c for c in ALL_SENSOR_COLS if c.endswith("_avg")]
else:
    SENSOR_COLS = list(ALL_SENSOR_COLS)

if DROP_REACTIVE:
    SENSOR_COLS = [c for c in SENSOR_COLS if not c.startswith("reactive_power_")]

print(f"{len(ALL_SENSOR_COLS)} sensor columns -> {len(SENSOR_COLS)} selected")

# ---- angle detection -------------------------------------------------------
# Prefer metadata; fall back to name prefixes. Always print what was found so
# the choice can be checked rather than trusted.
def detect_angle_sensors(feature_description, sensor_cols):
    angle_bases = set()
    text_cols = [c for c in feature_description.columns
                 if feature_description[c].dtype == object and c != "sensor_name"]
    pat = re.compile(r"\b(degree|degrees|deg|angle|direction|azimuth)\b", re.I)
    for _, row in feature_description.iterrows():
        blob = " ".join(str(row[c]) for c in text_cols)
        if pat.search(blob):
            angle_bases.add(str(row["sensor_name"]))
    if not angle_bases:  # metadata gave nothing -> name-based fallback
        name_pat = re.compile(r"^(wind_direction|nacelle_direction|nacelle_position|"
                              r"pitch_angle|blade_angle|yaw_angle|wind_dir)", re.I)
        angle_bases = {n for n in feature_description["sensor_name"].astype(str)
                       if name_pat.match(n)}
    angle_cols = [c for c in sensor_cols
                  if any(c == f"{b}_avg" or c.startswith(f"{b}_") for b in angle_bases)]
    return sorted(set(angle_cols))

ANGLE_COLS = detect_angle_sensors(feature_description, SENSOR_COLS) if ANGLE_TRANSFORM else []
LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
print(f"Angle columns detected ({len(ANGLE_COLS)}): {ANGLE_COLS if len(ANGLE_COLS) <= 12 else ANGLE_COLS[:12] + ['...']}")

# Sanity-check the detection: a true angle column spans roughly 0-360 (or
# -180..180). Anything outside that is a false positive and must NOT be sin/cos'd.
if ANGLE_COLS:
    _chk = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP,
                       usecols=ANGLE_COLS)
    _lo, _hi = _chk.min(), _chk.max()
    _suspect = [c for c in ANGLE_COLS
                if not ((-190 <= _lo[c] <= 5) and (170 <= _hi[c] <= 370))]
    print("\nAngle range check (first file):")
    for c in ANGLE_COLS:
        flag = "  <-- NOT angle-like" if c in _suspect else ""
        print(f"  {c:<22} [{_lo[c]:8.2f}, {_hi[c]:8.2f}]{flag}")
    if _suspect:
        print(f"\nDropping {len(_suspect)} false positives from the angle set.")
        ANGLE_COLS = [c for c in ANGLE_COLS if c not in _suspect]
        LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
    del _chk

FEATURE_NAMES = LINEAR_COLS + [f"{c}__sin" for c in ANGLE_COLS] + [f"{c}__cos" for c in ANGLE_COLS]
N_FEATURES = len(FEATURE_NAMES)
print(f"Model input dimension: {N_FEATURES}")

PRESENT_DESCRIPTIVE = [c for c in DESCRIPTIVE_COLS if c in _sample.columns]
READ_COLS = PRESENT_DESCRIPTIVE + SENSOR_COLS
print(f"Columns actually parsed per file: {len(READ_COLS)} of {len(_sample.columns)}")

POWER_COL = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("power_") and f"{b}_avg" in SENSOR_COLS), None)
WIND_COL  = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("wind_speed_") and f"{b}_avg" in SENSOR_COLS), None)
print(f"Power-curve filter uses: power={POWER_COL}  wind={WIND_COL}")

In [ ]:
# =============================================================================
# PARSED-DATA CACHE
# =============================================================================
# Each event CSV is ~55k rows x ~957 columns and takes 25-45s to parse. Arm A
# reads all 58 once; Arm B reads them twice. That is ~30-60 min of GPU-quota
# time during which the GPU is idle. Build the cache once in a CPU session
# (CPU sessions do not consume GPU quota), save /kaggle/working as a Kaggle
# Dataset, attach it to the GPU notebooks, and every later run skips parsing.

def resolve_cache_dir():
    """Prefer an attached cache dataset; fall back to the working dir."""
    for base in ("/kaggle/input", "/kaggle/working"):
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            if "care_cache_manifest.json" in filenames:
                return dirpath
    return os.path.join("/kaggle/working", "care_cache")

CACHE_DIR = resolve_cache_dir()
CACHE_MANIFEST = os.path.join(CACHE_DIR, "care_cache_manifest.json")
CACHE_READY = os.path.exists(CACHE_MANIFEST)

if CACHE_READY:
    _man = json.load(open(CACHE_MANIFEST))
    # A cache built for a different column set would silently change the model
    # input. Refuse it rather than train on the wrong features.
    if set(_man.get("sensor_cols", [])) != set(SENSOR_COLS):
        print(f"Cache at {CACHE_DIR} was built for a different feature set "
              f"({len(_man.get('sensor_cols', []))} cols vs {len(SENSOR_COLS)}). Ignoring it.")
        CACHE_READY = False
    else:
        print(f"Using cache: {CACHE_DIR}  ({_man['n_events']} events, "
              f"built {_man['built_at']})")
if not CACHE_READY:
    print(f"No usable cache. Parsing CSVs directly. "
          f"Run the cache-builder notebook to skip this next time.")

def cache_path(eid):
    return os.path.join(CACHE_DIR, f"event_{eid}.parquet")

def build_cache(out_dir=None, verbose=True):
    """Parse every event CSV once and write it as parquet. CPU session only."""
    import datetime
    out_dir = out_dir or os.path.join("/kaggle/working", "care_cache")
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    for i, fname in enumerate(csv_files, 1):
        eid = extract_dataset_id(fname)
        df = pd.read_csv(os.path.join(DATASETS_DIR, fname), sep=DATA_SEP,
                         usecols=READ_COLS,
                         dtype={c: DTYPE for c in SENSOR_COLS})
        df["time_stamp"] = pd.to_datetime(df["time_stamp"])
        df = df.sort_values("time_stamp").reset_index(drop=True)
        df.to_parquet(os.path.join(out_dir, f"event_{eid}.parquet"),
                      index=False, compression="snappy")
        if verbose:
            print(f"[{i:>2}/{len(csv_files)}] event {eid:<3} {len(df):>6} rows "
                  f"({time.time()-t0:.0f}s elapsed)")
    json.dump({"sensor_cols": SENSOR_COLS, "descriptive_cols": PRESENT_DESCRIPTIVE,
               "n_events": len(csv_files), "farm": FARM, "sep": DATA_SEP,
               "built_at": datetime.datetime.now().isoformat(timespec="seconds")},
              open(os.path.join(out_dir, "care_cache_manifest.json"), "w"), indent=2)
    print(f"\nCache written to {out_dir} in {(time.time()-t0)/60:.1f} min")
    return out_dir

## Build

In [ ]:
CACHE_OUT = build_cache()

## Verify
Re-read one event from the cache and confirm it matches the CSV.

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(CACHE_OUT, "*.parquet")))
print(f"{len(files)} parquet files, "
      f"{sum(os.path.getsize(f) for f in files)/1e6:.0f} MB total")

probe = csv_files[0]; eid = extract_dataset_id(probe)
from_cache = pd.read_parquet(os.path.join(CACHE_OUT, f"event_{eid}.parquet"))
from_csv = pd.read_csv(os.path.join(DATASETS_DIR, probe), sep=DATA_SEP,
                       usecols=READ_COLS, dtype={c: DTYPE for c in SENSOR_COLS})
from_csv["time_stamp"] = pd.to_datetime(from_csv["time_stamp"])
from_csv = from_csv.sort_values("time_stamp").reset_index(drop=True)

assert from_cache.shape == from_csv.shape, (from_cache.shape, from_csv.shape)
assert list(from_cache.columns) == list(from_csv.columns)
num = [c for c in SENSOR_COLS]
assert np.allclose(from_cache[num].to_numpy(), from_csv[num].to_numpy(),
                   equal_nan=True), "cache does not round-trip"
print(f"Round-trip verified on event {eid}: {from_cache.shape}")